# Gold (A): order flow

**Owner:** Person A  
**Builds:** `dim_customers`, `fact_orders`, `fact_order_items`, `mart_customer_rfm`

## 1. `gold.dim_customers`
- **Grain:** one row per `customer_id`.
- **Columns:** `customer_id`, `customer_unique_id`, `zip_code_prefix`, `city`, `state`.
- **Note:** a real person is `customer_unique_id` (one person can have several `customer_id` values). Use `DISTINCTCOUNT(customer_unique_id)` in Power BI for customer counts.
- **Checks:** `customer_id` unique; row count equals `silver.customers`.

## 2. `gold.fact_orders`
- **Grain:** one row per order.
- **Columns:** `order_id`, `customer_id`, `order_status`, `order_date` (DATE), `approved_date`, `delivered_carrier_date`, `delivered_customer_date`, `estimated_delivery_date`, `delivery_days`, `delay_days`, `is_late` (BIT), `total_payment`, `payment_installments_max`, `main_payment_type`.
- **Logic:**
  - Aggregate `silver.order_payments` by `order_id` first (sum of payment value, main payment type).
  - `delivery_days` = delivered date minus purchase date; `delay_days` = delivered date minus estimated date; `is_late` = `delay_days > 0`.
  - Delivery columns stay `NULL` for orders that were not delivered.
- **Checks:** `order_id` unique; row count equals `silver.orders`; total payment matches the sum in `silver.order_payments` (small differences should be explained in the log).

## 3. `gold.fact_order_items`
- **Grain:** one row per order item (`order_id` + `order_item_id`).
- **Columns:** `order_id`, `order_item_id`, `product_id`, `seller_id`, `customer_id`, `order_date`, `order_status`, `price`, `freight_value`, `item_total` (price + freight).
- **Note:** keep payments out of this table. Order-level amounts live in `fact_orders`, otherwise they repeat on every item and inflate totals.
- **Checks:** row count equals `silver.order_items`; `SUM(price)` equals the silver total.

## 4. `gold.mart_customer_rfm`
- **Grain:** one row per `customer_unique_id`.
- **Columns:** `customer_unique_id`, `recency_days`, `frequency` (distinct orders), `monetary` (sum of item prices), `r_score`, `f_score`, `m_score`, `segment`.
- **Logic:** reference date is the day after the last order in the data (not today). Scores 1-5 with `NTILE`/`qcut`; most customers order only once, so frequency scoring needs a simple rule (for example 1 order, 2 orders, 3+ orders). Document the segment rules.
- **Why a mart:** segmentation is awkward in DAX, so it is calculated once here.